In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams['font.family'] = 'Liberation Sans'
rcParams['font.size'] = 12
rcParams['figure.autolayout'] = True
rcParams["savefig.bbox"] = 'tight'
rcParams["savefig.format"] = 'png'
rcParams["savefig.dpi"] = 150
rcParams["grid.linestyle"] = '--'

# from matplotlib_inline import backend_inline
# backend_inline.set_matplotlib_formats('svg')

In [ ]:
model_path = "model_phasenet_das/PhaseNet-DAS-v1.pth"

In [6]:
import torch
from collections import OrderedDict

# Load the model checkpoint
checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)

# Check what's in the checkpoint
print("=" * 80)
print("CHECKPOINT KEYS:")
print("=" * 80)
for key in checkpoint.keys():
    print(f"  - {key}")
print()

# Model state dict
if 'model' in checkpoint:
    state_dict = checkpoint['model']
    print("=" * 80)
    print("MODEL STATE DICT (using 'model' key)")
    print("=" * 80)
elif 'state_dict' in checkpoint:
    state_dict = checkpoint['state_dict']
    print("=" * 80)
    print("MODEL STATE DICT (using 'state_dict' key)")
    print("=" * 80)
else:
    state_dict = checkpoint
    print("=" * 80)
    print("MODEL STATE DICT (direct checkpoint)")
    print("=" * 80)

# Display layer names and shapes
print("\nLayer Details:")
print("-" * 80)
total_params = 0
for name, param in state_dict.items():
    num_params = param.numel()
    total_params += num_params
    print(f"{name:60s} | Shape: {str(list(param.shape)):20s} | Params: {num_params:,}")

print("=" * 80)
print(f"Total Parameters: {total_params:,}")
print(f"Model Size: {total_params * 4 / (1024**2):.2f} MB (float32)")
print("=" * 80)


CHECKPOINT KEYS:
  - model
  - optimizer
  - lr_scheduler
  - epoch
  - args
  - scaler

MODEL STATE DICT (using 'model' key)

Layer Details:
--------------------------------------------------------------------------------
backbone.input_conv.enc1_conv1.weight                        | Shape: [16, 1, 7, 7]        | Params: 784
backbone.input_conv.enc1_norm1.weight                        | Shape: [16]                 | Params: 16
backbone.input_conv.enc1_norm1.bias                          | Shape: [16]                 | Params: 16
backbone.input_conv.enc1_norm1.running_mean                  | Shape: [16]                 | Params: 16
backbone.input_conv.enc1_norm1.running_var                   | Shape: [16]                 | Params: 16
backbone.input_conv.enc1_norm1.num_batches_tracked           | Shape: []                   | Params: 1
backbone.input_conv.enc1_conv2.weight                        | Shape: [16, 16, 7, 7]       | Params: 12,544
backbone.input_conv.enc1_norm2.weight        

In [7]:
# Additional checkpoint information
print("\n" + "=" * 80)
print("ADDITIONAL CHECKPOINT INFORMATION:")
print("=" * 80)

if 'epoch' in checkpoint:
    print(f"Epoch: {checkpoint['epoch']}")
    
if 'args' in checkpoint:
    print("\nTraining Arguments:")
    print(checkpoint['args'])
    
if 'config' in checkpoint:
    print("\nModel Config:")
    print(checkpoint['config'])

if 'optimizer' in checkpoint:
    print(f"\nOptimizer state: Present")
    
if 'scheduler' in checkpoint:
    print(f"Scheduler state: Present")



ADDITIONAL CHECKPOINT INFORMATION:
Epoch: 8

Training Arguments:
Namespace(data_path='gs://quakeflow_das', data_list=['results/training_v1//data.txt'], label_path=None, label_list=['results/training_v1//labels_train.txt'], noise_list=['results/training_v1//noise_train.txt'], hdf5_file=None, test_data_path='gs://quakeflow_das', test_data_list='results/training_v1//data.txt', test_label_path='+', test_label_list='results/training/labels_test.txt', test_noise_list='results/training/noise_test.txt', dataset='', model='phasenet_das', backbone='unet', device='cuda', compile=False, batch_size=4, epochs=10, workers=4, opt='adamw', lr=0.001, momentum=0.9, weight_decay=0.1, norm_weight_decay=None, bias_weight_decay=None, transformer_embedding_decay=None, clip_grad_norm=None, lr_warmup_epochs=1, lr_warmup_method='linear', lr_warmup_decay=0.01, lr_scheduler='cosineannealinglr', lr_step_size=8, lr_gamma=0.1, lr_min=0.0, print_freq=10, output_dir='model_phasenet_das', resume='True', start_epoch=0, 

In [8]:
# Build the model architecture to compare
import sys
sys.path.insert(0, '..')
from eqnet.models.phasenet_das import build_model

# Build the model with same config
model = build_model(in_channels=1, out_channels=3, reg=0.1)

# Load the weights
model.load_state_dict(state_dict)

print("\n" + "=" * 80)
print("MODEL ARCHITECTURE:")
print("=" * 80)
print(model)



MODEL ARCHITECTURE:
PhaseNetDAS(
  (backbone): UNet(
    (input_conv): Sequential(
      (enc1_conv1): Conv2d(1, 16, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3), bias=False)
      (enc1_norm1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (enc1_relu1): ReLU(inplace=True)
      (enc1_conv2): Conv2d(16, 16, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), bias=False)
      (enc1_norm2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (enc1_relu2): ReLU(inplace=True)
    )
    (encoder12): Sequential(
      (enc2_conv1): Conv2d(16, 32, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3), bias=False)
      (enc2_norm1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (enc2_relu1): ReLU(inplace=True)
      (enc2_conv2): Conv2d(32, 32, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), bias=False)
      (enc2_norm2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True,

In [10]:
# Model Summary
print("\n" + "=" * 80)
print("MODEL SUMMARY:")
print("=" * 80)
print(f"Model Name: PhaseNet-DAS")
print(f"Training Epoch: {checkpoint['epoch']}")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Model Size: {total_params * 4 / (1024**2):.2f} MB (float32)")
print()

# Count parameters by module
print("Parameters by Module:")
print("-" * 80)
backbone_params = sum(p.numel() for p in model.backbone.parameters())
classifier_params = sum(p.numel() for p in model.classifier.parameters())
print(f"  Backbone (UNet):     {backbone_params:>10,} params ({backbone_params/total_params*100:.1f}%)")
print(f"  Classifier (Head):   {classifier_params:>10,} params ({classifier_params/total_params*100:.1f}%)")
print()

# Input/Output configuration
print("Model Configuration:")
print("-" * 80)
print(f"  Input Channels:  {checkpoint['args'].nt} x {checkpoint['args'].nx} (Time x Space)")
print(f"  Crop Size:       {checkpoint['args'].crop_nt} x {checkpoint['args'].crop_nx}")
print(f"  Output Channels: 3 (Noise, P-wave, S-wave)")
print(f"  Phases Detected: {checkpoint['args'].phases}")
print(f"  Regularization:  {checkpoint['args'].reg}")
print("=" * 80)



MODEL SUMMARY:
Model Name: PhaseNet-DAS
Training Epoch: 8
Total Parameters: 10,684,016
Trainable Parameters: 10,681,027
Model Size: 40.76 MB (float32)

Parameters by Module:
--------------------------------------------------------------------------------
  Backbone (UNet):     10,678,672 params (99.9%)
  Classifier (Head):        2,355 params (0.0%)

Model Configuration:
--------------------------------------------------------------------------------
  Input Channels:  3072 x 5120 (Time x Space)
  Crop Size:       1024 x 1024
  Output Channels: 3 (Noise, P-wave, S-wave)
  Phases Detected: ['P', 'S']
  Regularization:  0.0


In [12]:
# Analyze layer structure
print("\n" + "=" * 80)
print("BACKBONE ARCHITECTURE (UNet):")
print("=" * 80)

# Group layers by type
layer_types = {}
for name, param in state_dict.items():
    if 'backbone' in name:
        layer_type = name.split('.')[1] if len(name.split('.')) > 1 else 'root'
        if layer_type not in layer_types:
            layer_types[layer_type] = []
        layer_types[layer_type].append((name, param.shape, param.numel()))

# Display grouped
for layer_type, layers in sorted(layer_types.items()):
    total_in_group = sum(n for _, _, n in layers)
    print(f"\n{layer_type}: {len(layers)} layers, {total_in_group:,} params")
    
print("\n" + "=" * 80)
print("CLASSIFIER ARCHITECTURE:")
print("=" * 80)
for name, param in state_dict.items():
    if 'classifier' in name:
        print(f"  {name:40s} {str(list(param.shape)):20s} {param.numel():,}")



BACKBONE ARCHITECTURE (UNet):

decoder21: 12 layers, 125,634 params

decoder32: 12 layers, 502,146 params

decoder43: 12 layers, 2,007,810 params

encoder12: 12 layers, 75,522 params

encoder23: 12 layers, 301,570 params

encoder34: 12 layers, 1,205,250 params

encoder45: 12 layers, 4,818,946 params

input_conv: 12 layers, 13,458 params

output_conv: 6 layers, 25,153 params

upconv54: 6 layers, 1,606,145 params

CLASSIFIER ARCHITECTURE:
  classifier.laplace_kernel                [1, 3, 3, 3]         27
  classifier.layers.weight                 [3, 16, 7, 7]        2,352
  classifier.layers.bias                   [3]                  3


In [15]:
# Test forward pass to verify dimensions
print("\n" + "=" * 80)
print("MODEL INPUT/OUTPUT TEST:")
print("=" * 80)

# Create dummy input with larger size (model requires padding space)
batch_size = 2
crop_nt = checkpoint['args'].crop_nt
crop_nx = checkpoint['args'].crop_nx

# Use larger size to accommodate padding requirements (moving_norm uses padding of 1024)
test_nt = crop_nt + 2048  # Add space for padding
test_nx = crop_nx + 2048

dummy_input = {
    'data': torch.randn(batch_size, 1, test_nt, test_nx)
}

# Set model to eval mode
model.eval()

# Forward pass
with torch.no_grad():
    output = model(dummy_input)

print(f"Input shape:  {dummy_input['data'].shape} (batch, channels, time, space)")
print(f"Output shape: {output['phase'].shape} (batch, classes, time, space)")
print(f"Output classes: {['Noise', 'P-wave', 'S-wave']}")
print()
print(f"Typical training crop size: {crop_nt} x {crop_nx}")
print(f"Full data size: {checkpoint['args'].nt} x {checkpoint['args'].nx}")
print()
print("Model successfully processes input with expected dimensions!")
print("=" * 80)



MODEL INPUT/OUTPUT TEST:
Input shape:  torch.Size([2, 1, 3072, 3072]) (batch, channels, time, space)
Output shape: torch.Size([2, 3, 3072, 3072]) (batch, classes, time, space)
Output classes: ['Noise', 'P-wave', 'S-wave']

Typical training crop size: 1024 x 1024
Full data size: 3072 x 5120

Model successfully processes input with expected dimensions!


In [16]:
# Final Comprehensive Summary
print("\n" + "🔍 " + "=" * 78)
print("COMPREHENSIVE MODEL ANALYSIS: PhaseNet-DAS-v1.pth")
print("=" * 80)
print()
print("📊 MODEL STATISTICS:")
print(f"   • Total Parameters:      {total_params:>12,}")
print(f"   • Trainable Parameters:  {sum(p.numel() for p in model.parameters() if p.requires_grad):>12,}")
print(f"   • Model Size:            {total_params * 4 / (1024**2):>12.2f} MB (float32)")
print(f"   • Training Epoch:        {checkpoint['epoch']:>12}")
print()
print("🏗️  ARCHITECTURE:")
print(f"   • Backbone: UNet ({backbone_params:,} params, {backbone_params/total_params*100:.1f}%)")
print(f"     - Encoder stages:   encoder12, encoder23, encoder34, encoder45")
print(f"     - Decoder stages:   decoder43, decoder32, decoder21")
print(f"     - Upconv:          upconv54")
print(f"     - Input/Output:    input_conv, output_conv")
print(f"   • Head: UNet Classifier ({classifier_params:,} params, {classifier_params/total_params*100:.1f}%)")
print(f"     - Conv layers with Laplace regularization")
print()
print("📐 INPUT/OUTPUT SPECIFICATIONS:")
print(f"   • Input:  (batch, 1, time, space)")
print(f"   • Output: (batch, 3, time, space)")
print(f"   • Classes: Noise, P-wave, S-wave")
print(f"   • Training crop size: {crop_nt} × {crop_nx}")
print(f"   • Full data size:     {checkpoint['args'].nt} × {checkpoint['args'].nx}")
print()
print("⚙️  TRAINING CONFIGURATION:")
print(f"   • Optimizer:        {checkpoint['args'].opt}")
print(f"   • Learning Rate:    {checkpoint['args'].lr}")
print(f"   • Batch Size:       {checkpoint['args'].batch_size}")
print(f"   • Total Epochs:     {checkpoint['args'].epochs}")
print(f"   • Regularization:   {checkpoint['args'].reg}")
print(f"   • Stack Noise:      {checkpoint['args'].stack_noise}")
print(f"   • Stack Event:      {checkpoint['args'].stack_event}")
print(f"   • Random Crop:      {checkpoint['args'].random_crop}")
print()
print("✅ MODEL STATUS: Successfully loaded and validated!")
print("=" * 80)



🔍 ==============================================================================
COMPREHENSIVE MODEL ANALYSIS: PhaseNet-DAS-v1.pth

📊 MODEL STATISTICS:
   • Total Parameters:        10,684,016
   • Trainable Parameters:    10,681,027
   • Model Size:                   40.76 MB (float32)
   • Training Epoch:                   8

🏗️  ARCHITECTURE:
   • Backbone: UNet (10,678,672 params, 99.9%)
     - Encoder stages:   encoder12, encoder23, encoder34, encoder45
     - Decoder stages:   decoder43, decoder32, decoder21
     - Upconv:          upconv54
     - Input/Output:    input_conv, output_conv
   • Head: UNet Classifier (2,355 params, 0.0%)
     - Conv layers with Laplace regularization

📐 INPUT/OUTPUT SPECIFICATIONS:
   • Input:  (batch, 1, time, space)
   • Output: (batch, 3, time, space)
   • Classes: Noise, P-wave, S-wave
   • Training crop size: 1024 × 1024
   • Full data size:     3072 × 5120

⚙️  TRAINING CONFIGURATION:
   • Optimizer:        adamw
   • Learning Rate:    0.001
